# creating aquifer wise training data

Sanitised source snapshot. See the repository README and WORKFLOW.md before running. Outputs and execution metadata are removed. Historical experiments are not an unattended pipeline or reproduced scientific result.


In [ ]:
import os
from pathlib import Path

REPO_ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "prepare_wells.py").is_file())
WORKSPACE_ROOT = Path(os.environ.get("CGWB_WORKSPACE", str(REPO_ROOT / "data" / "cgwb"))).expanduser().resolve()
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE_ROOT)


In [ ]:
import pandas as pd

In [ ]:
# Define the path to the pickle file
pickle_file_path = f"{WORKSPACE_ROOT}/9. Creating aquifer wise training data/input/mws_without_calculated_cwd.pickle"

# Read the pickle file
mws_without_calculated_cwd_df = pd.read_pickle(pickle_file_path)

In [ ]:
mws_without_calculated_cwd_df

In [ ]:
import pandas as pd
import os

# Define the input directory path containing the CSV files
csv_directory_path = f"{WORKSPACE_ROOT}/9. Creating aquifer wise training data/input/"

# List to store dataframes
dataframes = []

# Loop through each file in the directory and read the CSV files
for filename in os.listdir(csv_directory_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(csv_directory_path, filename)
        df = pd.read_csv(file_path)
        dataframes.append(df)

# Combine all the dataframes into one
combined_dataframe = pd.concat(dataframes, ignore_index=True)

In [ ]:
# Display the first few rows of the combined dataframe to check the result
combined_dataframe

In [ ]:
# Merge the dataframes on 'id_microwatershed' column
merged_cgwb_and_calculated_values = pd.merge(mws_without_calculated_cwd_df, combined_dataframe, on='id_micro_watershed', how='inner')

In [ ]:
# Display the first few rows of the merged dataframe to check the result
print(merged_cgwb_and_calculated_values)

In [ ]:
for column in merged_cgwb_and_calculated_values.columns:
    print(column)

# saving the dataframe for future analysis

In [ ]:
merged_cgwb_and_calculated_values.to_pickle(f'{WORKSPACE_ROOT}/9. Creating aquifer wise training data/output/analyze_merged_cgwb_and_calculated_values.pickle')

# Creating a row for every season and year

In [ ]:
# Melt the dataframe to long format
id_vars = [
    'stream_order_categorical', 'terrain_cluster', 'consecutive_non_nan_pairs', 'DN', 'id_micro_watershed',
    'geometry_micro_watershed', 'index_right', 'id_right', 'aquifer', 'aquifer0', 'aquifers', 'area_re',
    'avg_mbgl', 'm2_perday', 'm3_per_day', 'mbgl', 'newcode14', 'newcode43', 'objectid_right', 'pa_order',
    'per_cm', 'shape_Area', 'shape_Leng', 'state', 'system', 'test', 'yeild__', 'zone_m', 'aquifer_category',
    'percentage_area_stream_order_1', 'percentage_area_stream_order_2', 'percentage_area_stream_order_3',
    'percentage_area_stream_order_4', 'percentage_area_stream_order_5', 'percentage_area_stream_order_6',
    'percentage_area_stream_order_7', 'percentage_area_stream_order_8', 'percentage_area_stream_order_9',
    'percentage_area_stream_order_10', 'percentage_area_stream_order_11', 'coordinates'
]
value_vars = [col for col in merged_cgwb_and_calculated_values.columns if col not in id_vars]

melted_df = pd.melt(merged_cgwb_and_calculated_values, id_vars=id_vars, value_vars=value_vars, var_name='date_season_metric', value_name='value')

# Extract year, season, and metric from the 'date_season_metric' column
melted_df[['year', 'season', 'metric']] = melted_df['date_season_metric'].str.extract(r'(\d{4})_(Jan_March|April_June|Jul_Sept|Oct_Dec)_(.*)')

# Map season to season_numeric
season_mapping = {'Jan_March': 1, 'April_June': 2, 'Jul_Sept': 3, 'Oct_Dec': 4}
melted_df['season_numeric'] = melted_df['season'].map(season_mapping)

# Drop the 'date_season_metric' column
melted_df = melted_df.drop(columns=['date_season_metric'])

# Verify that id_vars does not contain list-like values before pivoting
for col in id_vars + ['year', 'season', 'season_numeric']:
    if melted_df[col].apply(lambda x: isinstance(x, list)).any():
        melted_df[col] = melted_df[col].apply(lambda x: str(x))

# Pivot the dataframe to have separate columns for each metric
final_df = melted_df.pivot_table(index=id_vars + ['year', 'season', 'season_numeric'], columns='metric', values='value').reset_index()

# Rename columns accordingly
final_df.columns = [col[0] if isinstance(col, tuple) else col for col in final_df.columns]
final_df.columns = [col.replace('_calculated_', '_') for col in final_df.columns]
final_df.columns = [col.replace('_GT_CWD', 'CWD_GT') if '_GT_CWD' in col else col for col in final_df.columns]

# Sort the dataframe by 'id_micro_watershed', 'year', and 'season_numeric'
final_df = final_df.sort_values(by=['id_micro_watershed', 'year', 'season_numeric'])

In [ ]:
final_df

In [ ]:
for column in final_df.columns:
    print(column)

In [ ]:
# Display unique values in the 'terrain_cluster' column of final_df
unique_terrain_clusters = final_df['terrain_cluster'].unique()
unique_terrain_clusters

In [ ]:
# Drop the columns 'index_right' and 'id_right' from final_df
final_df = final_df.drop(columns=['objectid_right','index_right', 'id_right'])
final_df

In [ ]:
# Extracting unique values for the specified columns
unique_values = {
    'DN': final_df['DN'].unique(),
    'area_re': final_df['area_re'].unique(),
    'avg_mbgl': final_df['avg_mbgl'].unique(),
    'm2_perday': final_df['m2_perday'].unique(),
    'm3_per_day': final_df['m3_per_day'].unique(),
    'mbgl': final_df['mbgl'].unique(),
    'newcode14': final_df['newcode14'].unique(),
    'newcode43': final_df['newcode43'].unique(),
    'pa_order': final_df['pa_order'].unique(),
    'per_cm': final_df['per_cm'].unique(),
    'shape_Area': final_df['shape_Area'].unique(),
    'shape_Leng': final_df['shape_Leng'].unique(),
    'state': final_df['state'].unique(),
    'system': final_df['system'].unique(),
    'test': final_df['test'].unique(),
    'yeild__': final_df['yeild__'].unique(),
    'zone_m': final_df['zone_m'].unique()
}

# Finding values that aquifer, aquifer0, aquifers can take for a given aquifer_category
aquifer_values_by_category = final_df.groupby('aquifer_category')[['aquifer', 'aquifer0', 'aquifers']].apply(lambda x: x.apply(lambda y: y.unique()))

unique_values, aquifer_values_by_category


In [ ]:
final_df.columns

In [ ]:
lagged_data=final_df.copy()

# Add lagged precipitation values using groupby and shift
for k in range(1,13):
  lagged_data[f'n-{k}_season_calculated_WellDepth'] = final_df.groupby('id_micro_watershed')['calculated_WellDepth'].shift(k)

for k in range(1,4):
  lagged_data[f'n-{k}_season_calculated_Precipitation'] = final_df.groupby('id_micro_watershed')['calculated_Precipitation'].shift(k)



# Display the first few rows to verify the changes
lagged_data.head()

In [ ]:
for columns in lagged_data.columns:
  print(columns)

In [ ]:
lagged_data

In [ ]:
lagged_data['coordinates']

In [ ]:
# Display the first few entries of the 'coordinates' column to inspect their structure
print(lagged_data['coordinates'].head(10))

In [ ]:
import ast

# Function to convert string representations of lists back to actual lists
def convert_to_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return [None, None]
    return x

# Apply the function to ensure all entries are lists
lagged_data['coordinates'] = lagged_data['coordinates'].apply(convert_to_list)

# Split the 'coordinates' column into 'y_coordinate' and 'x_coordinate'
lagged_data[['y_coordinate', 'x_coordinate']] = pd.DataFrame(lagged_data['coordinates'].tolist(), index=lagged_data.index)

# Drop the original 'coordinates' column
lagged_data = lagged_data.drop(columns=['coordinates'])

# Ensure the new columns are stored as 1D values
lagged_data['y_coordinate'] = lagged_data['y_coordinate'].apply(lambda x: x if isinstance(x, float) else x)
lagged_data['x_coordinate'] = lagged_data['x_coordinate'].apply(lambda x: x if isinstance(x, float) else x)

# Display the first few rows to verify the changes
lagged_data.head()

In [ ]:
for column in lagged_data.columns:
    print(column)

In [ ]:
# List of columns to keep
columns_to_keep = [
    'stream_order_categorical', 'terrain_cluster', 'id_micro_watershed', 'geometry_micro_watershed', 'aquifer_category',
    'percentage_area_stream_order_1', 'percentage_area_stream_order_2', 'percentage_area_stream_order_3', 'percentage_area_stream_order_4',
    'percentage_area_stream_order_5', 'percentage_area_stream_order_6', 'percentage_area_stream_order_7', 'percentage_area_stream_order_8',
    'percentage_area_stream_order_9', 'percentage_area_stream_order_10', 'percentage_area_stream_order_11', 'year', 'season', 'season_numeric',
    'GT_CWD', 'calculated_DeltaG', 'calculated_ET', 'calculated_G', 'calculated_Precipitation', 'calculated_RunOff', 'calculated_WellDepth',
    'n-1_season_calculated_WellDepth', 'n-2_season_calculated_WellDepth', 'n-3_season_calculated_WellDepth', 'n-4_season_calculated_WellDepth',
    'n-5_season_calculated_WellDepth', 'n-6_season_calculated_WellDepth', 'n-7_season_calculated_WellDepth', 'n-8_season_calculated_WellDepth',
    'n-9_season_calculated_WellDepth', 'n-10_season_calculated_WellDepth', 'n-11_season_calculated_WellDepth', 'n-12_season_calculated_WellDepth',
    'n-1_season_calculated_Precipitation', 'n-2_season_calculated_Precipitation', 'n-3_season_calculated_Precipitation', 'y_coordinate', 'x_coordinate'
]

# Filter the DataFrame
filtered_lagged_data = lagged_data[columns_to_keep]

filtered_lagged_data

In [ ]:
for column in filtered_lagged_data.columns:
    print(column)

In [ ]:
import pandas as pd

# Define seasonal precipitation conditions
seasonal_precipitation_conditions = {
    1: ['n-1_season_calculated_Precipitation', 'n-2_season_calculated_Precipitation'],
    2: ['n-1_season_calculated_Precipitation', 'n-2_season_calculated_Precipitation', 'n-3_season_calculated_Precipitation'],
    3: [],
    4: ['n-1_season_calculated_Precipitation']
}

# Define aquifer-specific lagged GT conditions
aquifer_memory_conditions = {
    'Crystalline Systems': 4,
    'Volcanic Systems': 4,
    'Sedimentary Hard Rock': 8,
    'Himalayan Systems': 8,
    'Alluvial': 12,
    'Sedimentary Soft Rock': 12
}

# Function to filter data based on seasonal precipitation and aquifer memory conditions
def filter_season_aquifer_data(df):
    filtered_data = []

    for season, precip_columns in seasonal_precipitation_conditions.items():
        for aquifer, memory in aquifer_memory_conditions.items():
            print(f"\nProcessing Season: {season}, Aquifer: {aquifer}")

            # Define the lagged well depth columns for the aquifer
            lagged_well_depth_columns = [f'n-{i}_season_calculated_WellDepth' for i in range(1, memory + 1)]
            print(f"Lagged Well Depth Columns: {lagged_well_depth_columns}")

            # Combine the required columns
            columns_to_select = [
                'id_micro_watershed', 'geometry_micro_watershed', 'aquifer_category', 'stream_order_categorical',
                'terrain_cluster', 'percentage_area_stream_order_1', 'percentage_area_stream_order_2',
                'percentage_area_stream_order_3', 'percentage_area_stream_order_4', 'percentage_area_stream_order_5',
                'percentage_area_stream_order_6', 'percentage_area_stream_order_7', 'percentage_area_stream_order_8',
                'percentage_area_stream_order_9', 'percentage_area_stream_order_10', 'percentage_area_stream_order_11',
                'year', 'season', 'season_numeric', 'GT_CWD', 'calculated_DeltaG', 'calculated_ET', 'calculated_G',
                'calculated_Precipitation', 'calculated_RunOff', 'calculated_WellDepth', 'y_coordinate', 'x_coordinate'
            ] + precip_columns + lagged_well_depth_columns

            # Ensure only columns that exist in the DataFrame are selected
            columns_to_select = [col for col in columns_to_select if col in df.columns]
            print(f"Columns to Select: {columns_to_select}")

            # Filter the data for the current season and aquifer
            season_aquifer_data = df[(df['season_numeric'] == season) & (df['aquifer_category'] == aquifer)][columns_to_select]
            print(f"Filtered Data Shape (before dropping inappropriate columns): {season_aquifer_data.shape}")

            # Remove inappropriate lagged precipitation columns
            inappropriate_precip_columns = [f'n-{i}_season_calculated_Precipitation' for i in range(3, memory + 1)]
            print(f"Inappropriate Precipitation Columns: {inappropriate_precip_columns}")

            for col in inappropriate_precip_columns:
                if col in season_aquifer_data.columns:
                    season_aquifer_data.drop(columns=[col], inplace=True)
                    print(f"Dropped Column: {col}")

            print(f"Filtered Data Shape (after dropping inappropriate columns): {season_aquifer_data.shape}")
            print(f"Final Columns: {season_aquifer_data.columns.tolist()}")

            # Append the filtered data to the list
            filtered_data.append(season_aquifer_data)

    # Concatenate the filtered data into a single DataFrame
    result_df = pd.concat(filtered_data, ignore_index=True)
    return result_df

# Apply the function to the filtered_lagged_data
season_aquifer_filtered_data = filter_season_aquifer_data(filtered_lagged_data)

# Function to print columns for each season and aquifer combination
def print_columns_for_each_season_aquifer(df):
    for season, precip_columns in seasonal_precipitation_conditions.items():
        for aquifer in aquifer_memory_conditions.keys():
            # Filter the data for the current season and aquifer
            season_aquifer_data = df[(df['season_numeric'] == season) & (df['aquifer_category'] == aquifer)]

            # Print the season, aquifer, and columns
            print(f"\nSeason: {season}, Aquifer: {aquifer}")
            print(season_aquifer_data.columns)
            print()

# Apply the function to the filtered data
print_columns_for_each_season_aquifer(season_aquifer_filtered_data)


In [ ]:
print(season_aquifer_filtered_data)

In [ ]:
# Function to print columns for each season and aquifer combination
def print_columns_for_each_season_aquifer(df):
    for season, precip_columns in seasonal_precipitation_conditions.items():
        for aquifer in aquifer_memory_conditions.keys():
            # Filter the data for the current season and aquifer
            season_aquifer_data = df[(df['season_numeric'] == season) & (df['aquifer_category'] == aquifer)]

            # Print the season, aquifer, and columns
            print(f"Season: {season}, Aquifer: {aquifer}")
            print(season_aquifer_data.columns)
            print()

# Apply the function to the filtered data
print_columns_for_each_season_aquifer(season_aquifer_filtered_data)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

# Continuous, categorical, and ordinal columns
continuous_columns = [
    'percentage_area_stream_order_1', 'percentage_area_stream_order_2', 'percentage_area_stream_order_3',
    'percentage_area_stream_order_4', 'percentage_area_stream_order_5', 'percentage_area_stream_order_6',
    'percentage_area_stream_order_7', 'percentage_area_stream_order_8', 'percentage_area_stream_order_9',
    'percentage_area_stream_order_10', 'percentage_area_stream_order_11', 'calculated_ET',
    'calculated_Precipitation', 'calculated_RunOff', 'calculated_WellDepth', 'y_coordinate', 'x_coordinate'
]
categorical_columns = ['terrain_cluster']
ordinal_columns = ['stream_order_categorical', 'year']

# Create a list of ordinal categories for stream_order_categorical and year
ordinal_categories = [
    sorted(filtered_lagged_data['stream_order_categorical'].unique()),
    list(range(filtered_lagged_data['year'].min(), filtered_lagged_data['year'].max() + 1))
]

# Preprocess and split the data
def preprocess_data(df, season, aquifer):
    additional_covariates = seasonal_precipitation_conditions[season]
    lagged_gt_columns = [f'n-{i}_season_calculated_WellDepth' for i in range(1, aquifer_memory_conditions[aquifer] + 1)]
    columns_to_use = continuous_columns + ordinal_columns + categorical_columns + additional_covariates + lagged_gt_columns

    X = df[columns_to_use].apply(pd.to_numeric, errors='coerce')
    y = df['GT_CWD'].apply(pd.to_numeric, errors='coerce')
    groups = df['id_micro_watershed']

    # Remove rows with NaN values in the target variable
    non_nan_indices = y.dropna().index
    X = X.loc[non_nan_indices]
    y = y.loc[non_nan_indices]
    groups = groups.loc[non_nan_indices]

    # Remove rows with NaN values in features
    X_non_nan_indices = X.dropna().index
    X = X.loc[X_non_nan_indices]
    y = y.loc[X_non_nan_indices]
    groups = groups.loc[X_non_nan_indices]

    # Preprocessing for numerical data
    numerical_transformer = StandardScaler()

    # Preprocessing for categorical data
    categorical_transformer = OneHotEncoder(handle_unknown='ignore')

    # Preprocessing for ordinal data
    ordinal_transformer = OrdinalEncoder(categories=ordinal_categories)

    # Bundle preprocessing for numerical, categorical, and ordinal data
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, continuous_columns + additional_covariates + lagged_gt_columns),
            ('cat', categorical_transformer, categorical_columns),
            ('ord', ordinal_transformer, ordinal_columns)
        ])

    # Create preprocessing pipeline
    clf = Pipeline(steps=[('preprocessor', preprocessor)])

    # Preprocess the features
    X_preprocessed = clf.fit_transform(X)

    return X_preprocessed, y, clf, groups, columns_to_use

# Filter and preprocess data based on seasonal precipitation and aquifer memory conditions
def preprocess_season_aquifer_data(df):
    preprocessed_data = {}

    for season, precip_columns in seasonal_precipitation_conditions.items():
        for aquifer, memory in aquifer_memory_conditions.items():
            lagged_gt_columns = [f'n-{i}_season_calculated_WellDepth' for i in range(1, memory + 1)]
            columns_to_select = [
                'id_micro_watershed', 'geometry_micro_watershed', 'aquifer_category', 'stream_order_categorical',
                'terrain_cluster', 'percentage_area_stream_order_1', 'percentage_area_stream_order_2',
                'percentage_area_stream_order_3', 'percentage_area_stream_order_4', 'percentage_area_stream_order_5',
                'percentage_area_stream_order_6', 'percentage_area_stream_order_7', 'percentage_area_stream_order_8',
                'percentage_area_stream_order_9', 'percentage_area_stream_order_10', 'percentage_area_stream_order_11',
                'year', 'season', 'season_numeric', 'GT_CWD', 'calculated_DeltaG', 'calculated_ET', 'calculated_G',
                'calculated_Precipitation', 'calculated_RunOff', 'calculated_WellDepth', 'y_coordinate', 'x_coordinate'
            ] + precip_columns + lagged_gt_columns

            season_aquifer_data = df[(df['season_numeric'] == season) & (df['aquifer_category'] == aquifer)][columns_to_select]
            X_preprocessed, y, clf, groups, columns_to_use = preprocess_data(season_aquifer_data, season, aquifer)
            preprocessed_data[(season, aquifer)] = (X_preprocessed, y, clf, groups, columns_to_use)

    return preprocessed_data

# Apply the function to the filtered_lagged_data
preprocessed_season_aquifer_data = preprocess_season_aquifer_data(filtered_lagged_data)

# Split the data for each season and aquifer combination
split_data_dict = {}
for key, data in preprocessed_season_aquifer_data.items():
    X_preprocessed, y, preprocessor, groups, columns_to_use = data
    gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
    train_idx, test_idx = next(gss.split(X_preprocessed, y, groups))
    X_train, X_test = X_preprocessed[train_idx], X_preprocessed[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    split_data_dict[key] = (X_train, X_test, y_train, y_test, preprocessor, columns_to_use)


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

# Continuous, categorical, and ordinal columns
continuous_columns = [
    'percentage_area_stream_order_1', 'percentage_area_stream_order_2', 'percentage_area_stream_order_3',
    'percentage_area_stream_order_4', 'percentage_area_stream_order_5', 'percentage_area_stream_order_6',
    'percentage_area_stream_order_7', 'percentage_area_stream_order_8', 'percentage_area_stream_order_9',
    'percentage_area_stream_order_10', 'percentage_area_stream_order_11', 'calculated_ET',
    'calculated_Precipitation', 'calculated_RunOff', 'calculated_WellDepth', 'y_coordinate', 'x_coordinate'
]
categorical_columns = ['terrain_cluster']
ordinal_columns = ['stream_order_categorical', 'year']

# Create a list of ordinal categories for stream_order_categorical and year
ordinal_categories = [
    sorted(filtered_lagged_data['stream_order_categorical'].unique().astype(str)),  # Ensure strings
    list(range(int(filtered_lagged_data['year'].min()), int(filtered_lagged_data['year'].max()) + 1))  # Ensure integers
]

# Preprocess and split the data
def preprocess_data(df, season, aquifer):
    additional_covariates = seasonal_precipitation_conditions[season]
    lagged_gt_columns = [f'n-{i}_season_calculated_WellDepth' for i in range(1, aquifer_memory_conditions[aquifer] + 1)]
    columns_to_use = continuous_columns + ordinal_columns + categorical_columns + additional_covariates + lagged_gt_columns

    X = df[columns_to_use].apply(pd.to_numeric, errors='coerce')
    y = df['GT_CWD'].apply(pd.to_numeric, errors='coerce')
    groups = df['id_micro_watershed']

    # Remove rows with NaN values in the target variable
    non_nan_indices = y.dropna().index
    X = X.loc[non_nan_indices]
    y = y.loc[non_nan_indices]
    groups = groups.loc[non_nan_indices]

    # Remove rows with NaN values in features
    X_non_nan_indices = X.dropna().index
    X = X.loc[X_non_nan_indices]
    y = y.loc[X_non_nan_indices]
    groups = groups.loc[X_non_nan_indices]

    # Preprocessing for numerical data
    numerical_transformer = StandardScaler()

    # Preprocessing for categorical data
    categorical_transformer = OneHotEncoder(handle_unknown='ignore')

    # Preprocessing for ordinal data
    ordinal_transformer = OrdinalEncoder(categories=ordinal_categories)

    # Bundle preprocessing for numerical, categorical, and ordinal data
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, continuous_columns + additional_covariates + lagged_gt_columns),
            ('cat', categorical_transformer, categorical_columns),
            ('ord', ordinal_transformer, ordinal_columns)
        ])

    # Create preprocessing pipeline
    clf = Pipeline(steps=[('preprocessor', preprocessor)])

    # Preprocess the features
    X_preprocessed = clf.fit_transform(X)

    return X_preprocessed, y, clf, groups, columns_to_use

# Filter and preprocess data based on seasonal precipitation and aquifer memory conditions
def preprocess_season_aquifer_data(df):
    preprocessed_data = {}

    for season, precip_columns in seasonal_precipitation_conditions.items():
        for aquifer, memory in aquifer_memory_conditions.items():
            lagged_gt_columns = [f'n-{i}_season_calculated_WellDepth' for i in range(1, memory + 1)]
            columns_to_select = [
                'id_micro_watershed', 'geometry_micro_watershed', 'aquifer_category', 'stream_order_categorical',
                'terrain_cluster', 'percentage_area_stream_order_1', 'percentage_area_stream_order_2',
                'percentage_area_stream_order_3', 'percentage_area_stream_order_4', 'percentage_area_stream_order_5',
                'percentage_area_stream_order_6', 'percentage_area_stream_order_7', 'percentage_area_stream_order_8',
                'percentage_area_stream_order_9', 'percentage_area_stream_order_10', 'percentage_area_stream_order_11',
                'year', 'season', 'season_numeric', 'GT_CWD', 'calculated_DeltaG', 'calculated_ET', 'calculated_G',
                'calculated_Precipitation', 'calculated_RunOff', 'calculated_WellDepth', 'y_coordinate', 'x_coordinate'
            ] + precip_columns + lagged_gt_columns

            season_aquifer_data = df[(df['season_numeric'] == season) & (df['aquifer_category'] == aquifer)][columns_to_select]
            X_preprocessed, y, clf, groups, columns_to_use = preprocess_data(season_aquifer_data, season, aquifer)
            preprocessed_data[(season, aquifer)] = (X_preprocessed, y, clf, groups, columns_to_use)

    return preprocessed_data

# Apply the function to the filtered_lagged_data
preprocessed_season_aquifer_data = preprocess_season_aquifer_data(filtered_lagged_data)

# Split the data for each season and aquifer combination
split_data_dict = {}
for key, data in preprocessed_season_aquifer_data.items():
    X_preprocessed, y, preprocessor, groups, columns_to_use = data
    gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
    train_idx, test_idx = next(gss.split(X_preprocessed, y, groups))
    X_train, X_test = X_preprocessed[train_idx], X_preprocessed[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    split_data_dict[key] = (X_train, X_test, y_train, y_test, preprocessor, columns_to_use)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

# Continuous, categorical, and ordinal columns
continuous_columns = [
    'percentage_area_stream_order_1', 'percentage_area_stream_order_2', 'percentage_area_stream_order_3',
    'percentage_area_stream_order_4', 'percentage_area_stream_order_5', 'percentage_area_stream_order_6',
    'percentage_area_stream_order_7', 'percentage_area_stream_order_8', 'percentage_area_stream_order_9',
    'percentage_area_stream_order_10', 'percentage_area_stream_order_11', 'calculated_ET',
    'calculated_Precipitation', 'calculated_RunOff', 'calculated_WellDepth', 'y_coordinate', 'x_coordinate'
]
categorical_columns = ['terrain_cluster']
ordinal_columns = ['stream_order_categorical', 'year']

# Create a list of ordinal categories for stream_order_categorical and year
ordinal_categories = [
    sorted(filtered_lagged_data['stream_order_categorical'].unique()),
    list(range(filtered_lagged_data['year'].min(), filtered_lagged_data['year'].max() + 1))
]

# Preprocess and split the data
def preprocess_data(df, season, aquifer):
    additional_covariates = seasonal_precipitation_conditions[season]
    lagged_gt_columns = [f'n-{i}_season_calculated_WellDepth' for i in range(1, aquifer_memory_conditions[aquifer] + 1)]
    columns_to_use = continuous_columns + ordinal_columns + categorical_columns + additional_covariates + lagged_gt_columns

    X = df[columns_to_use].apply(pd.to_numeric, errors='coerce')
    y = df['GT_CWD'].apply(pd.to_numeric, errors='coerce')
    groups = df['id_micro_watershed']

    # Remove rows with NaN values in the target variable
    non_nan_indices = y.dropna().index
    X = X.loc[non_nan_indices]
    y = y.loc[non_nan_indices]
    groups = groups.loc[non_nan_indices]

    # Remove rows with NaN values in features
    X_non_nan_indices = X.dropna().index
    X = X.loc[X_non_nan_indices]
    y = y.loc[X_non_nan_indices]
    groups = groups.loc[X_non_nan_indices]

    # Preprocessing for numerical data
    numerical_transformer = StandardScaler()

    # Preprocessing for categorical data
    categorical_transformer = OneHotEncoder(handle_unknown='ignore')

    # Preprocessing for ordinal data
    ordinal_transformer = OrdinalEncoder(categories=ordinal_categories)

    # Bundle preprocessing for numerical, categorical, and ordinal data
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, continuous_columns + additional_covariates + lagged_gt_columns),
            ('cat', categorical_transformer, categorical_columns),
            ('ord', ordinal_transformer, ordinal_columns)
        ])

    # Create preprocessing pipeline
    clf = Pipeline(steps=[('preprocessor', preprocessor)])

    # Preprocess the features
    X_preprocessed = clf.fit_transform(X)

    # Retrieve the feature names
    feature_names = (
        preprocessor.named_transformers_['num'].get_feature_names_out(continuous_columns + additional_covariates + lagged_gt_columns).tolist() +
        preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_columns).tolist() +
        ordinal_columns
    )

    return X_preprocessed, y.reset_index(drop=True), clf, groups.reset_index(drop=True), feature_names

# Filter and preprocess data based on seasonal precipitation and aquifer memory conditions
def preprocess_season_aquifer_data(df):
    preprocessed_data = {}

    for season, precip_columns in seasonal_precipitation_conditions.items():
        for aquifer, memory in aquifer_memory_conditions.items():
            lagged_gt_columns = [f'n-{i}_season_calculated_WellDepth' for i in range(1, memory + 1)]
            columns_to_select = [
                'id_micro_watershed', 'geometry_micro_watershed', 'aquifer_category', 'stream_order_categorical',
                'terrain_cluster', 'percentage_area_stream_order_1', 'percentage_area_stream_order_2',
                'percentage_area_stream_order_3', 'percentage_area_stream_order_4', 'percentage_area_stream_order_5',
                'percentage_area_stream_order_6', 'percentage_area_stream_order_7', 'percentage_area_stream_order_8',
                'percentage_area_stream_order_9', 'percentage_area_stream_order_10', 'percentage_area_stream_order_11',
                'year', 'season', 'season_numeric', 'GT_CWD', 'calculated_DeltaG', 'calculated_ET', 'calculated_G',
                'calculated_Precipitation', 'calculated_RunOff', 'calculated_WellDepth', 'y_coordinate', 'x_coordinate'
            ] + precip_columns + lagged_gt_columns

            season_aquifer_data = df[(df['season_numeric'] == season) & (df['aquifer_category'] == aquifer)][columns_to_select]
            X_preprocessed, y, clf, groups, feature_names = preprocess_data(season_aquifer_data, season, aquifer)
            preprocessed_data[(season, aquifer)] = (X_preprocessed, y, clf, groups, feature_names)

    return preprocessed_data

# Apply the function to the filtered_lagged_data
preprocessed_season_aquifer_data = preprocess_season_aquifer_data(filtered_lagged_data)

# Split the data for each season and aquifer combination
split_data_dict = {}
for key, data in preprocessed_season_aquifer_data.items():
    X_preprocessed, y, preprocessor, groups, feature_names = data
    gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
    train_idx, test_idx = next(gss.split(X_preprocessed, y, groups))
    X_train, X_test = X_preprocessed[train_idx], X_preprocessed[test_idx]
    y_train, y_test = y.iloc[train_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)
    split_data_dict[key] = (X_train, X_test, y_train, y_test, preprocessor, feature_names)


In [ ]:
# Function to train linear regression model and provide summary with feature names
def train_linear_regression(X_train, y_train, X_test, y_test, feature_names):
    # Ensure consistency of columns between X_train and X_test
    X_train_df = pd.DataFrame(X_train, columns=feature_names)
    X_test_df = pd.DataFrame(X_test, columns=feature_names)

    # Add a constant to the training and testing data for the intercept
    X_train_const = sm.add_constant(X_train_df, has_constant='add')
    X_test_const = sm.add_constant(X_test_df, has_constant='add')

    # Debug: print shapes and columns
    print(f"X_train_const columns: {X_train_const.columns}")
    print(f"X_test_const columns: {X_test_const.columns}")
    print(f"Length of feature_names: {len(feature_names)}")
    print(f"Shape of X_train_const: {X_train_const.shape}")
    print(f"Shape of X_test_const: {X_test_const.shape}")

    # Fit the model
    model = sm.OLS(y_train, X_train_const).fit()

    # Predict
    y_train_pred = model.predict(X_train_const)
    y_test_pred = model.predict(X_test_const)

    # Adjust the summary to include feature names
    model.model.data.xnames = X_train_const.columns.tolist()

    return model, y_train_pred, y_test_pred


In [ ]:
# Train and evaluate the model
linear_regression_results = {}
for key, (X_train, X_test, y_train, y_test, preprocessor, feature_names) in split_data_dict.items():
    model, y_train_pred, y_test_pred = train_linear_regression(X_train, y_train, X_test, y_test, feature_names)
    linear_regression_results[key] = (model, y_train_pred, y_test_pred, y_train, y_test)


In [ ]:
# Function to plot results
def plot_results(y_train, y_train_pred, y_test, y_test_pred):
    fig, axs = plt.subplots(1, 2, figsize=(12, 6))

    axs[0].scatter(y_train, y_train_pred, alpha=0.3)
    axs[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], '--r')
    axs[0].set_xlabel('Actual')
    axs[0].set_ylabel('Predicted')
    axs[0].set_title('Train Data')

    axs[1].scatter(y_test, y_test_pred, alpha=0.3)
    axs[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r')
    axs[1].set_xlabel('Actual')
    axs[1].set_ylabel('Predicted')
    axs[1].set_title('Test Data')

    plt.show()


# Evaluate results
for key, (model, y_train_pred, y_test_pred, y_train, y_test) in linear_regression_results.items():
    print(f"Results for Season {key[0]}, Aquifer {key[1]}:")
    print(model.summary())
    print(f"Train RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred))}")
    print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred))}")
    plot_results(y_train, y_train_pred, y_test, y_test_pred)

In [ ]:
def plot_results(y_train, y_train_pred, y_test, y_test_pred, model_name, season, aquifer):
    fig, axs = plt.subplots(1, 2, figsize=(12, 6))

    axs[0].scatter(y_train, y_train_pred, alpha=0.3)
    axs[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], '--r')
    axs[0].set_xlabel('Actual')
    axs[0].set_ylabel('Predicted')
    axs[0].set_title(f'{model_name} - Train Data\nSeason: {season}, Aquifer: {aquifer}')

    axs[1].scatter(y_test, y_test_pred, alpha=0.3)
    axs[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r')
    axs[1].set_xlabel('Actual')
    axs[1].set_ylabel('Predicted')
    axs[1].set_title(f'{model_name} - Test Data\nSeason: {season}, Aquifer: {aquifer}')

    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def train_and_evaluate_model(ModelClass, model_params, X_train, y_train, X_test, y_test, model_name, season, aquifer):
    model = ModelClass(**model_params)
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    # Compute Normalized RMSE (NRMSE)
    train_nrmse_mean = train_rmse / np.mean(y_train)
    test_nrmse_mean = test_rmse / np.mean(y_test)
    train_nrmse_std = train_rmse / np.std(y_train)
    test_nrmse_std = test_rmse / np.std(y_test)
    train_nrmse_range = train_rmse / (np.max(y_train) - np.min(y_train))
    test_nrmse_range = test_rmse / (np.max(y_test) - np.min(y_test))

    print(f"{model_name} Train RMSE: {train_rmse}")
    print(f"{model_name} Test RMSE: {test_rmse}")
    print(f"{model_name} Train MAE: {train_mae}")
    print(f"{model_name} Test MAE: {test_mae}")
    print(f"{model_name} Train R²: {train_r2}")
    print(f"{model_name} Test R²: {test_r2}")
    print(f"{model_name} Train NRMSE (Mean): {train_nrmse_mean}")
    print(f"{model_name} Test NRMSE (Mean): {test_nrmse_mean}")
    print(f"{model_name} Train NRMSE (Std): {train_nrmse_std}")
    print(f"{model_name} Test NRMSE (Std): {test_nrmse_std}")
    print(f"{model_name} Train NRMSE (Range): {train_nrmse_range}")
    print(f"{model_name} Test NRMSE (Range): {test_nrmse_range}")

    plot_results(y_train, y_train_pred, y_test, y_test_pred, model_name, season, aquifer)

    return model, y_train_pred, y_test_pred, train_rmse, test_rmse, train_mae, test_mae, train_r2, test_r2, \
           train_nrmse_mean, test_nrmse_mean, train_nrmse_std, test_nrmse_std, train_nrmse_range, test_nrmse_range


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn_params = {
    'n_neighbors': 5
}


In [ ]:
from sklearn.svm import SVR

svr_params = {
    'kernel': 'rbf',
    'C': 1.0,
    'epsilon': 0.1
}


In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt_params = {
    'random_state': 42
}

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_params = {
    'n_estimators': 100,
    'random_state': 42
}


In [ ]:
import pandas as pd

results = []

In [ ]:
# Apply the models to each dataset in split_data_dict
for key, (X_train, X_test, y_train, y_test, preprocessor, feature_names) in split_data_dict.items():
    season, aquifer = key
    print(f"Evaluating models for season {season} and aquifer {aquifer}")

    # KNN Regression
    model, y_train_pred, y_test_pred, train_rmse, test_rmse, train_mae, test_mae, train_r2, test_r2, \
    train_nrmse_mean, test_nrmse_mean, train_nrmse_std, test_nrmse_std, train_nrmse_range, test_nrmse_range = train_and_evaluate_model(
        KNeighborsRegressor, knn_params, X_train, y_train, X_test, y_test, "KNN Regression", season, aquifer)
    results.append([season, aquifer, 'KNN Regression', knn_params, train_rmse, test_rmse, train_mae, test_mae, train_r2, test_r2,
                    train_nrmse_mean, test_nrmse_mean, train_nrmse_std, test_nrmse_std, train_nrmse_range, test_nrmse_range])

    # SVM Regression
    model, y_train_pred, y_test_pred, train_rmse, test_rmse, train_mae, test_mae, train_r2, test_r2, \
    train_nrmse_mean, test_nrmse_mean, train_nrmse_std, test_nrmse_std, train_nrmse_range, test_nrmse_range = train_and_evaluate_model(
        SVR, svr_params, X_train, y_train, X_test, y_test, "SVM Regression", season, aquifer)
    results.append([season, aquifer, 'SVM Regression', svr_params, train_rmse, test_rmse, train_mae, test_mae, train_r2, test_r2,
                    train_nrmse_mean, test_nrmse_mean, train_nrmse_std, test_nrmse_std, train_nrmse_range, test_nrmse_range])

    # Decision Tree Regression
    model, y_train_pred, y_test_pred, train_rmse, test_rmse, train_mae, test_mae, train_r2, test_r2, \
    train_nrmse_mean, test_nrmse_mean, train_nrmse_std, test_nrmse_std, train_nrmse_range, test_nrmse_range = train_and_evaluate_model(
        DecisionTreeRegressor, dt_params, X_train, y_train, X_test, y_test, "Decision Tree Regression", season, aquifer)
    results.append([season, aquifer, 'Decision Tree Regression', dt_params, train_rmse, test_rmse, train_mae, test_mae, train_r2, test_r2,
                    train_nrmse_mean, test_nrmse_mean, train_nrmse_std, test_nrmse_std, train_nrmse_range, test_nrmse_range])

    # Random Forest Regression
    model, y_train_pred, y_test_pred, train_rmse, test_rmse, train_mae, test_mae, train_r2, test_r2, \
    train_nrmse_mean, test_nrmse_mean, train_nrmse_std, test_nrmse_std, train_nrmse_range, test_nrmse_range = train_and_evaluate_model(
        RandomForestRegressor, rf_params, X_train, y_train, X_test, y_test, "Random Forest Regression", season, aquifer)
    results.append([season, aquifer, 'Random Forest Regression', rf_params, train_rmse, test_rmse, train_mae, test_mae, train_r2, test_r2,
                    train_nrmse_mean, test_nrmse_mean, train_nrmse_std, test_nrmse_std, train_nrmse_range, test_nrmse_range])


In [ ]:
# Create a DataFrame to store the results
results_df = pd.DataFrame(results, columns=[
    'Season', 'Aquifer', 'Model', 'Parameters', 'Train RMSE', 'Test RMSE',
    'Train MAE', 'Test MAE', 'Train R²', 'Test R²', 'Train NRMSE (Mean)',
    'Test NRMSE (Mean)', 'Train NRMSE (Std)', 'Test NRMSE (Std)', 'Train NRMSE (Range)', 'Test NRMSE (Range)'
])

# Display the results
results_df.head()


In [ ]:
results_df

In [ ]:
for column in results_df.columns:
  print(column)

In [ ]:
# Determine the best-performing model for each aquifer and season based on Test RMSE
best_models = results_df.loc[results_df.groupby(['Season', 'Aquifer'])['Test RMSE'].idxmin()]

# Display the best models
print("Best-performing models for each aquifer and season:")
print(best_models[['Season', 'Aquifer', 'Model', 'Test RMSE']])


In [ ]:
# Determine the best-performing model for each aquifer and season based on Test RMSE
best_models = results_df.loc[results_df.groupby(['Season', 'Aquifer'])['Train R²'].idxmin()]

# Display the best models
print("Best-performing models for each aquifer and season:")
print(best_models[['Season', 'Aquifer', 'Model', 'Train R²']])


In [ ]:
# Determine the best-performing model for each aquifer and season based on Test RMSE
best_models = results_df.loc[results_df.groupby(['Season', 'Aquifer'])['Test NRMSE (Std)'].idxmin()]

# Display the best models
print("Best-performing models for each aquifer and season:")
print(best_models[['Season', 'Aquifer', 'Model', 'Test NRMSE (Std)']])


In [ ]:
def plot_results(y_train, y_train_pred, y_test, y_test_pred):
    fig, axs = plt.subplots(1, 2, figsize=(12, 6))

    axs[0].scatter(y_train, y_train_pred, alpha=0.3)
    axs[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], '--r')
    axs[0].set_xlabel('Actual')
    axs[0].set_ylabel('Predicted')
    axs[0].set_title('Train Data')

    axs[1].scatter(y_test, y_test_pred, alpha=0.3)
    axs[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r')
    axs[1].set_xlabel('Actual')
    axs[1].set_ylabel('Predicted')
    axs[1].set_title('Test Data')

    plt.show()

# Evaluate results
for key, (model, y_train_pred, y_test_pred, y_train, y_test) in linear_regression_results.items():
    print(f"Results for Season {key[0]}, Aquifer {key[1]}:")
    print(model.summary())
    print(f"Train RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred))}")
    print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred))}")
    plot_results(y_train, y_train_pred, y_test, y_test_pred)
